# Lab 07 · A framework for choosing and checking

**Prepared by Mustafa Assaf** · 15–20 minute lab

## Run the same loop in a framework

**So far:** we wrote each workflow by hand. In Lab 06, a loop proposed a decision, had it checked, and retried once on a failure.

**New:** **LangGraph**, a framework that names the parts of a workflow. *Nodes* are steps, *edges* set their order, and the *state* is the data that the nodes share. A *conditional edge* picks the next node from the state. That is how a graph loops back.

Our graph has two nodes. `choose` asks the model for a decision, and `check` tests it against the policy. After `check`, a router either ends the run or sends it back to `choose` with feedback. The framework does not make the model's choice correct. It makes each step and each retry visible.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'langgraph']):
    %pip install -q "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Describe the state

`State` lists every field the nodes can read or write. `total=False` means a run may start with only some of them. We start each run with just the project. The scripted replies work as in Lab 06: one per attempt.

In [ ]:
from typing import TypedDict

class State(TypedDict, total=False):
    project: dict      # input: the project facts
    decision: str      # written by choose
    attempts: int      # written by choose
    passed: bool       # written by check
    feedback: str      # written by check, read by choose on a retry

projects = [
    {"name": "Course portal",     "team_size": 3,  "independent_deploys": False},
    {"name": "Payments platform", "team_size": 12, "independent_deploys": True},
    {"name": "Internal wiki",     "team_size": 9,  "independent_deploys": False},
    {"name": "Mobile backend",    "team_size": 5,  "independent_deploys": True},
]
scripted = {"Course portal": ["modular monolith"], "Payments platform": ["microservices"],
            "Internal wiki": ["microservices", "modular monolith"], "Mobile backend": ["modular monolith"]}

## Step 2 · Write the two nodes

A node is a function that receives the current state and returns only the fields it changes. LangGraph merges those fields into the state. Because a node is a plain function, you can test it on its own, as the last line does. `choose` counts its attempts. `check` writes a verdict, and on a failure it adds feedback that `choose` will see on the next attempt.

In [ ]:
def choose(state):
    """Ask the model for a decision; on a retry, include the checker's feedback."""
    project, attempts = state["project"], state.get("attempts", 0)
    prompt = f"Pick modular monolith or microservices for {project}. Reply with the label only."
    if state.get("feedback"):
        prompt += f"\nYour last answer failed a check: {state['feedback']} Answer again."
    reply = ask(prompt, example=scripted[project["name"]][attempts])
    return {"decision": reply.strip().rstrip(".").lower(), "attempts": attempts + 1}

def check(state):
    """Compare the decision with the course policy and explain a failure."""
    p = state["project"]
    expected = "microservices" if p["team_size"] >= 8 and p["independent_deploys"] else "modular monolith"
    if state["decision"] == expected:
        return {"passed": True, "feedback": ""}
    return {"passed": False, "feedback": f"The policy requires {expected} for these facts."}

print(check({"project": projects[2], "decision": "microservices"}))  # a node is a plain function

## Step 3 · Connect the nodes into a graph

The normal edges run `START → choose → check`. After `check`, `route` returns `"done"` or `"retry"`, and the dictionary maps each answer to the next node. The two-attempt limit lives in `route`, so the graph always ends. Printing the edges shows the structure that LangGraph built.

In [ ]:
from langgraph.graph import StateGraph, START, END

def route(state):
    """Finish on a pass or after two attempts; otherwise go back to choose."""
    return "done" if state["passed"] or state["attempts"] >= 2 else "retry"

graph = StateGraph(State)
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_edge(START, "choose")
graph.add_edge("choose", "check")
graph.add_conditional_edges("check", route, {"retry": "choose", "done": END})
app = graph.compile()
for edge in app.get_graph().edges:
    when = f" (when {edge.data})" if edge.conditional else ""
    print(f"{edge.source} -> {edge.target}{when}")

## Step 4 · Run the graph on every project

`app.invoke` runs the graph from `START` to `END` and returns the final state. We run it once per project and print a small report.

In [ ]:
results = [app.invoke({"project": project}) for project in projects]
for r in results:
    print(f"{r['project']['name']:<18} {r['decision']:<17} attempts: {r['attempts']}  passed: {r['passed']}")
print("Model calls:", sum(r["attempts"] for r in results))
assert all(r["passed"] for r in results), "A project still fails the policy check after two attempts."

## What you should see

Step 2 prints the failing check for the Internal wiki. Step 3 prints four edges, including `check -> choose (when retry)` and `check -> __end__ (when done)`. In Step 4, three projects pass on the first attempt. The Internal wiki takes two attempts: the first scripted reply, `microservices`, fails the check, and the retry passes. That makes `Model calls: 5`.

## Try it

1. Make the Internal wiki's second scripted reply `"microservices"` as well. The graph stops after two attempts, and the final check fails.
2. Allow three attempts: change `>= 2` to `>= 3` in `route`, and give the Internal wiki a third scripted reply.
3. Add a fifth project to `projects` and `scripted`. The graph needs no change.
4. In live mode, how many attempts does each project take?

## Takeaway

A framework names the parts that you wrote by hand in earlier labs: state, steps, order, and the decision to retry. Generation and checking stay in separate nodes, so a failure is easy to see and to limit. You have now used rules, prompts, evaluation, retrieval, tools, agent loops, skills, agent-to-agent review, and a framework, all on one small question.

## Read further

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangGraph quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart)
- [LangGraph Complete Guide](https://github.com/mkassaf/langgraph-complete-guide): continue here after this lab.